# Multi-channel teacher → student training (Thermal + Radar)

Run the cells top to bottom. Every cell states what it verified, because each
check here stands for a failure this project actually hit: a Drive mount that
cannot be remounted, a mount serving hours-old code, a helper tool that
corrupted a source file, an hour of GPU time spent before noticing the two
students were being taught to answer "person" always.

`both` is deliberately **not** one step: the thermal student takes about an
hour and the radar about twenty minutes, and a failure in the second used to
throw away the first. They are separate cells, and a two-minute preflight
proves the whole path before either is started.

**Runtime → Change runtime type → GPU** first (A100 is preferred).

## 1 · Drive

In [ ]:
import os, ast, shutil, subprocess, sys, time, json, glob

DRIVE = '/content/drive'
EXPORT_NAME = 'v2'  # Drive folder under thermal-fusion/gexport/
EPOCHS = 50
BATCH_SIZE = 32
WORKERS = 2
REMOTE = f'{DRIVE}/MyDrive/thermal-fusion/gexport/{EXPORT_NAME}'
DATA = f'/content/{EXPORT_NAME}'
OUT    = REMOTE + '/models'

def mount_drive(force=False):
    from google.colab import drive
    if force:
        try:
            drive.flush_and_unmount()
        except Exception as exc:
            print('unmount:', exc)
    try:
        drive.mount(DRIVE, force_remount=force)
    except ValueError as exc:
        # "Mountpoint must not already contain files" - a dead mount leaves
        # its directory behind and no plain mount() can get past it.
        print('mount:', exc, '-> forcing')
        drive.flush_and_unmount()
        drive.mount(DRIVE, force_remount=True)

if not os.path.exists(REMOTE + '/manifest.json'):
    mount_drive()
assert os.path.exists(REMOTE + '/manifest.json'), (
    'Drive unreachable. Runtime > Disconnect and delete runtime, rerun.')
print('drive OK ->', REMOTE)

## 2 · GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0),
      '| %.0f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 3 · Sync the training code — and prove it arrived

The Drive mount happily serves a cached copy of a file that changed on Drive
minutes ago, so copying is not enough: this re-copies, checks the source
parses, and checks the features this run depends on are actually present. If
the copy is stale it remounts and tries again rather than training yesterday's
model.

In [ ]:
REQUIRED = {
    'LABEL_NEGATIVE import': 'LABEL_NEGATIVE, LABEL_POSITIVE',
    'empty-frame weighting': 'negative_weight',
    'ambient augmentation' : 'augment_thermal_seq',
}

def sync_code(attempt=1):
    shutil.rmtree(DATA + '/code', ignore_errors=True)
    shutil.copytree(REMOTE + '/code', DATA + '/code')
    src = open(DATA + '/code/perception/students.py').read()
    try:
        ast.parse(src)                      # a half-written or patched file
    except SyntaxError as exc:              # must never reach the trainer
        print('students.py does not parse:', exc)
        return None, src
    missing = [k for k, needle in REQUIRED.items() if needle not in src]
    return missing, src

missing, src = sync_code()
if missing is None or missing:
    print('stale or broken copy - remounting and retrying')
    mount_drive(force=True)
    missing, src = sync_code(2)

assert missing is not None, 'students.py still does not parse - stop here'
assert not missing, 'missing from the copied code: %s' % missing
print('code OK | %d bytes' % len(src))
for name in REQUIRED:
    print('  present:', name)

## 4 · Dataset to local disk

Training reads every shard once. Doing that across the Drive mount is what
killed the first runs (`Transport endpoint is not connected` mid-epoch), so
the set is copied to the machine's own disk first.

In [ ]:
if not os.path.exists(DATA + '/manifest.json'):
    t0 = time.time()
    # DATA already exists because sync_code() created DATA/code. Copy the
    # directory contents, not REMOTE itself (which would create DATA/v2).
    subprocess.run(['cp', '-a', REMOTE + '/.', DATA + '/'], check=True)
    print('copied in %.1f min' % ((time.time() - t0) / 60))
    missing, _ = sync_code()          # cp brought code/ too - re-verify
    assert not missing, missing
else:
    print('data already local')

man = json.load(open(DATA + '/manifest.json'))
print('sessions:', len(man['sessions']), '| shards:',
      len(glob.glob(DATA + '/*.npz')))
print('train:', man['split']['train'])
print('val  :', man['split']['val'])

## 5 · Class balance

The trap both students fell into: with far more person-frames than
verified-empty ones, the cheapest route to a low loss is to answer "person"
always. It scores a perfect F1 on a val split with no empty frames, and paints
boxes on bare walls in the field. Training now weights the empty frames - this
is the ratio it will use, and the val split has to contain empty sessions or
the resulting numbers mean nothing again.

In [ ]:
import numpy as np, collections
train_sessions = set(man['split']['train'])
for plane in ('thermal', 'radar'):
    c = collections.Counter()
    for f in glob.glob(DATA + '/*.npz'):
        if f.split('/')[-1].rsplit('-', 1)[0] not in train_sessions:
            continue
        for v in np.load(f)[plane + '_label_state']:
            c[int(v)] += 1
    print('%-8s %6d person / %5d verified-empty -> weight %.1f'
          % (plane, c[1], c[0], min(c[1] / max(c[0], 1), 20.0)))
    assert c[0] > 0, (
        plane + ': no verified-empty frames in training - record an empty '
        'session and export it with --verified-negative')

neg_val = [s for s in man['split']['val']
           if man['sessions'][s]['verified_negative_frames'] > 0]
print('val sessions with empty frames:', neg_val)
assert neg_val, 'val has no empty session - false positives cannot be measured'

## 6 · Preflight — one epoch of each, ~2 minutes

Everything after this costs about ninety minutes. One epoch per student walks
the identical path (loaders, loss, evaluation, checkpoint write) and fails in
two minutes instead of ninety if anything is wrong.

In [ ]:
def train(student, epochs, out_dir, tag):
    env = dict(os.environ)
    env['PYTHONPATH'] = DATA + '/code'
    cmd = [sys.executable, '-u', '-m', 'perception.train_students',
           '--data', DATA, '--out', out_dir, '--student', student,
           '--epochs', str(epochs), '--batch-size', str(BATCH_SIZE),
           '--workers', str(WORKERS)]
    print(tag, '->', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    print('=== %s exit code: %d ===' % (tag, p.returncode))
    return p.returncode

PRE = '/content/preflight'
shutil.rmtree(PRE, ignore_errors=True)
rc = train('both', 1, PRE, 'PREFLIGHT')
assert rc == 0, 'preflight failed - the traceback is above; do not start the real run'
for name in ('thermal_student.pt', 'radar_student.pt'):
    assert os.path.exists(PRE + '/' + name), 'preflight wrote no ' + name
print('\npreflight OK - both students train, evaluate and save')

## 7 · Thermal student (~1 h)

Checkpoints land on **Drive** after every epoch, so a dropped runtime resumes:
re-run this cell and it continues from the last epoch instead of restarting.

In [ ]:
rc = train('thermal', EPOCHS, OUT, 'THERMAL')
assert rc == 0, 'thermal training failed - re-run this cell to resume'
assert os.path.exists(OUT + '/thermal_student.pt')
print('thermal_student.pt saved to Drive')

## 8 · Radar student (~20 min)

In [ ]:
rc = train('radar', EPOCHS, OUT, 'RADAR')
assert rc == 0, 'radar training failed - re-run this cell to resume'
assert os.path.exists(OUT + '/radar_student.pt')
print('radar_student.pt saved to Drive')

## 9 · Results

`false_positive_rate` is the number that was missing before: it is measured on
held-out sessions containing no people at all, one of them a dark empty room.
A high recall next to a high false-positive rate is the yes-man again.

In [ ]:
for name in ('thermal_student.pt', 'radar_student.pt'):
    path = OUT + '/' + name
    if not os.path.exists(path):
        print(name, 'missing'); continue
    ck = torch.load(path, map_location='cpu', weights_only=False)
    m = ck.get('metrics', {})
    print('===', name, '| trained', ck.get('created_utc'))
    print('    val:', ck.get('val_sessions'))
    print('    precision %.3f  recall %.3f  F1 %.3f'
          % (m.get('precision', float('nan')), m.get('recall', float('nan')),
             m.get('f1', float('nan'))))
    print('    false-positive rate %.3f   <- the one to watch'
          % m.get('false_positive_rate', float('nan')))
    print('    median error  u %.1f px  v %.1f px\n'
          % (m.get('median_u_px', float('nan')),
             m.get('median_v_px', float('nan'))))

## 10 · Export to ONNX for the Jetson

Writes `*_student.onnx` beside the checkpoints on Drive. Thermal is always
exportable. Radar is exported when the trained contract is point-cloud-only;
a checkpoint that learned RA/RD/RP is kept as `.pt` because the current
Jetson TensorRT runner does not yet accept those dense inputs.

In [ ]:
!pip -q install onnx onnxscript onnxruntime
LOCAL_MODELS = DATA + '/models'
os.makedirs(LOCAL_MODELS, exist_ok=True)
for name in ('thermal_student.pt', 'radar_student.pt'):
    source = OUT + '/' + name
    if os.path.exists(source):
        shutil.copy2(source, LOCAL_MODELS + '/' + name)
env = dict(os.environ); env['PYTHONPATH'] = DATA + '/code'
rc = subprocess.run([sys.executable, '-m', 'perception.export_students_onnx',
                     '--data', DATA], env=env, cwd=DATA).returncode
assert rc == 0, 'ONNX export failed'
for f in glob.glob(DATA + '/models/*.onnx*'):
    shutil.copy(f, OUT); print('->', os.path.basename(f))
print(sorted(os.listdir(OUT)))